In [1]:
import tensorflow as tf
import numpy as np
from degann.equations.system_ode import SystemODE
from degann.equations.equation_utils import system_ode_from_string, extract_iv
from sklearn.model_selection import train_test_split
import pandas as pd

In [2]:
f = "y0**2-2*y0+1 y0(0)=0.5"
f_prep = system_ode_from_string(f)
f_prep

[['y0**2-2*y0+1', 'y0(0)=0.5']]

In [3]:
f = SystemODE()
f.prepare_equations(1, f_prep)
f.solve((0, 5), 1000)

In [4]:
solution_table = pd.DataFrame(f.build_table(), columns=[["time_points", "values"]])
X, y = solution_table["time_points"], solution_table["values"]

In [5]:
solution_table.head()

,time_points,values
0,0.000000,0.500000
1,0.005005,0.501183
2,0.010010,0.502366
3,0.015015,0.503549
4,0.020020,0.504732


In [6]:
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=24)

# Ветка main

In [7]:
from degann.search_algorithms import random_search, simulated_annealing
from degann.networks.topology.tf_densenet import TensorflowDenseNet

## Случайный поиск

In [8]:
result_loss, result_epoch, result_loss_name, result_optimizer, result_nn = random_search(
    input_size=1,
    output_size=1,
    data=(X_train, y_train),
    opt="SGD",
    loss="MeanSquaredError",
    iterations=25,
    min_epoch=10,
    max_epoch=20,
    val_data=(X_valid, y_valid),
    nn_min_length=1,
    nn_max_length=5,
    nn_alphabet=["0a", "f8", "42", "56"],
)

In [9]:
result_nn

{'net_type': 'MyDense',
 'name': 'net',
 'input_size': 1,
 'block_size': [12, 8, 13, 12],
 'output_size': 1,
 'layer': [{'shape': 12,
   'inp_size': 1,
   'weights': [[-0.3102288842201233,
     -0.5910967588424683,
     -0.7580443024635315,
     0.2382337599992752,
     -0.7016312479972839,
     0.10764753818511963,
     -0.6753289699554443,
     0.8480365872383118,
     -0.13393032550811768,
     -0.9399520754814148,
     0.4905024468898773,
     -0.13621403276920319]],
   'biases': [-0.05619159713387489,
    0.38845589756965637,
    -0.5572699904441833,
    0.7230311036109924,
    -0.642485499382019,
    -0.9562637805938721,
    0.7706781029701233,
    -0.6668983697891235,
    -0.6939634084701538,
    -0.975682258605957,
    -0.15495048463344574,
    -0.4123131334781647],
   'layer_type': 'TensorflowDense',
   'dtype': 'float32',
   'activation': 'gelu',
   'decorator_params': None},
  {'shape': 8,
   'inp_size': 12,
   'weights': [[0.6563342809677124,
     0.46352002024650574,
     

Попробуем предсказать значение в точке 0.0, известно, что f(0.0) = 0.5

In [55]:
nn = TensorflowDenseNet(block_size=[1])
nn.from_dict(result_nn)
nn.call(tf.constant([[0.0]]))

<tf.Tensor: shape=(1, 1), dtype=float32, numpy=array([[0.5146367]], dtype=float32)>

### Описание

На каждой итерации происходит:
1. Обновление генератора случайных чисел.
2. Генерация случайной точки в пространстве параметров:
   - Случайным образом выбирается глубина сети (k), где min_length <= k <= max_length.
   - Случайным образом создаются слои: выбирается случайный элемент из nn_alphabet.
   - Случайным образом выбирается количество эпох (t), где min_epoch <= t <= max_epoch.
3. По полученным параметрам создаётся и обучается НС.
4. Если значение loss текущей НС меньше текущего лучшего значения, то обновляются best_epoch, best_net и best_loss.

Функция возвращает: best_loss, best_epoch, loss, opt, best_net.

## Алгоритм имитации отжига

In [57]:
result_loss, result_epoch, result_loss_name, result_optimizer, result_nn, final_iteration = simulated_annealing(
    input_size=1,
    output_size=1,
    data=(X_train, y_train),
    opt="Adam",
    loss="MeanSquaredError",
    threshold=1,
    min_epoch=10,
    max_epoch=20,
    max_iter=10,
    nn_min_length=1,
    nn_max_length=3,
    nn_alphabet=["0a", "f8", "42", "56"],
)

In [58]:
nn.from_dict(result_nn)
nn.call(tf.constant([[0.0]]))

<tf.Tensor: shape=(1, 1), dtype=float32, numpy=array([[0.50519156]], dtype=float32)>

### Описание

1. Генерируется случайная точка в пространстве параметров.
2. Если не передана стартовая НС, то она создаётся из параметров, полученных на шаге 1, иначе используется переданная НС. Количество эпох берётся из шага 1.
3. Полученная НС обучается, и её значение функции потерь сохраняется как лучшее.
4. Цикл while. Условия остановки: количество итераций >= max_iter или текущее значение loss <= заданный порог.
   - Обновляется генератор случайных чисел.
   - Вычисляется новое значение температуры (T) в зависимости от текущей итерации (k), максимального числа итераций (max_iter) и текущей температуры (t). Для вычисления используется одна из двух функций: temperature_lin, где T = 1 - (k + 1) / k_max, или temperature_exp, где T = T * alpha.
   - Вычисляется радиус поиска соседа в пространстве параметров (distance). Он вычисляется либо как константа (distance_const), либо по формуле distance = offset + temperature * multiplier (distance_lin).
   - Генерируется соседняя точка. Во внутреннем цикле while случайным образом изменяется текущая конфигурация НС (выполняется одно из доступных действий: изменение количества эпох, добавление/удаление слоя, изменение функции активации или числа нейронов). После каждого действия из distance вычитается расстояние от исходной НС до НС, полученной в результате изменения. Цикл завершается, если distance <= 0 или случайно с вероятностью 75% после каждой итерации.
   - По полученным параметрам создаётся и обучается новая НС. Если значение функции потерь новой НС меньше текущего или выполняется условие math.e ** ((curr_loss - neighbor_loss) / t) > random.random(), то происходит переход к новой НС как текущей.
   - Если значение loss текущей НС меньше предыдущего лучшего значения, то обновляются значения best_epoch, best_net и best_loss.

Функция возвращает: best_loss, best_epoch, loss, opt, best_nn, k (номер последней итерации).

# Ветка dev (grid_search)

In [ ]:
from degann.search_algorithms import grid_search


from degann.search_algorithms.search_algorithms_parameters import (
    BaseSearchParameters,
    GridSearchParameters,
)

from degann.networks.topology.densenet.compile_config import DenseNetCompileParams
from degann.networks.topology.densenet.topology_config import DenseNetParams
from degann.networks.topology.gan.gan import GAN 
from degann.networks.topology.gan.topology_config import GANTopologyParams
from degann.networks.topology.gan.compile_config import GANCompileParams

from degann.networks.topology.tuning_utils import FieldMetadata

In [ ]:
generator_metadata = {
    "block_size": FieldMetadata(
        value_range=(10, 20, 10),
        length_boundary=(1, 1),
    ),
}
generator_cfg = DenseNetParams(metadata=generator_metadata, activation_func="relu")

discriminator_metadata = {
    "block_size": FieldMetadata(value_range=(10, 20, 10), length_boundary=(1, 2)),
}
discriminator_cfg = DenseNetParams(
    metadata=discriminator_metadata, input_size=2, activation_func="relu"
)

GAN_config = GANTopologyParams(
    generator_params=generator_cfg, discriminator_params=discriminator_cfg
)

generator_compile_config = DenseNetCompileParams(
    optimizer="Adam", loss_func="BinaryCrossentropy", metric_funcs=[]
)
compile_metadata = {
    "optimizer": FieldMetadata(choices=["SGD", "Adam"]),
}
discriminator_compile_config = DenseNetCompileParams(
    metadata=compile_metadata,
    loss_func="BinaryCrossentropy",
    metric_funcs=[],
)
GAN_compile_cfg = GANCompileParams(
    generator_params=generator_compile_config,
    discriminator_params=discriminator_compile_config,
)

search_alg_params = BaseSearchParameters()
search_alg_params.model_cfg = GAN_config
search_alg_params.compile_cfg = GAN_compile_cfg
search_alg_params.data = (X_train, y_train)
search_alg_params.val_data = (X_valid, y_valid)

grid_search_parameters = GridSearchParameters(search_alg_params)
grid_search_parameters.min_epoch = 5
grid_search_parameters.max_epoch = 10
grid_search_parameters.epoch_step = 5

(
    result_metric_value,
    result_epoch,
    result_loss_name,
    result_optimizer,
    result_nn,
) = grid_search(grid_search_parameters)

print(result_nn)

{'generator': {'net_type': 'TFDense', 'name': '', 'input_size': 1, 'block_size': [20], 'output_size': 1, 'layer': [{'shape': 20, 'inp_size': 1, 'weights': [[0.1054505929350853, -1.0568443536758423, 0.9896453619003296, -0.09405499696731567, 0.19241414964199066, 0.2760612964630127, 0.5267316102981567, 0.5555077195167542, -0.29970717430114746, -0.7052760124206543, -0.7200982570648193, 0.4401833117008209, 0.22040367126464844, 0.0019350051879882812, -0.755558967590332, -0.5083143711090088, -0.7112116813659668, 0.4448399841785431, 0.6291581392288208, -0.8987948894500732]], 'biases': [0.5964310169219971, -0.06321820616722107, -0.05901256576180458, 0.9046491384506226, 0.2203841507434845, -0.7096168994903564, -0.19038768112659454, 0.9329661130905151, -0.5169413089752197, -0.7923617362976074, -0.2716386318206787, -0.5756074786186218, -0.6881711483001709, -0.51902174949646, -0.33686161041259766, -0.771817684173584, 0.8487924337387085, 0.6895444989204407, 0.2738281190395355, -0.5029919147491455], 'layer_type': 'TensorflowDense', 'dtype': 'float32', 'activation': 'relu', 'decorator_params': None}], 'out_layer': {'shape': 1, 'inp_size': 20, 'weights': [[-0.706920862197876], [-0.0631307065486908], [-0.7817713618278503], [0.20435026288032532], [-0.7094910144805908], [0.027073144912719727], [-0.09259898960590363], [0.789314329624176], [-0.9221072196960449], [0.26703834533691406], [0.92868971824646], [-0.46664443612098694], [-0.9353048801422119], [-0.4723057746887207], [-0.937220573425293], [-0.17502236366271973], [-0.17460104823112488], [0.4445388913154602], [0.820153534412384], [-0.5794088840484619]], 'biases': [-0.44476649165153503], 'layer_type': 'TensorflowDense', 'dtype': 'float32', 'activation': 'relu', 'decorator_params': None}}, 'discriminator': {'net_type': 'TFDense', 'name': '', 'input_size': 2, 'block_size': [10], 'output_size': 1, 'layer': [{'shape': 10, 'inp_size': 2, 'weights': [[0.012902718968689442, 0.9684498906135559, 0.7019546031951904, -0.37994953989982605, -0.9002600312232971, -1.0225186347961426, -0.3027927875518799, -0.16031989455223083, 0.11822211742401123, 0.031147241592407227], [0.13578712940216064, 0.26540979743003845, 0.39255326986312866, -0.4786532521247864, -0.15623879432678223, 0.02857443317770958, 0.60178542137146, 0.5391037464141846, 0.2885602116584778, -0.37982845306396484]], 'biases': [-0.013457098975777626, 0.7851802110671997, 0.4799230992794037, 1.0638355016708374, 0.2313491404056549, 0.6517014503479004, -0.07190355658531189, 0.8608404994010925, -0.7677075862884521, -0.45801782608032227], 'layer_type': 'TensorflowDense', 'dtype': 'float32', 'activation': 'relu', 'decorator_params': None}], 'out_layer': {'shape': 1, 'inp_size': 10, 'weights': [[-0.04677513614296913], [0.801634669303894], [-0.1675741970539093], [-0.9122216701507568], [-0.42605310678482056], [0.34166795015335083], [-0.10686688125133514], [-1.0213481187820435], [-0.9557952284812927], [0.6330878734588623]], 'biases': [0.5130225419998169], 'layer_type': 'TensorflowDense', 'dtype': 'float32', 'activation': 'relu', 'decorator_params': None}}}

## Описание

На каждой итерации происходит:
1. Перебор всех конфигураций модели из заданного пространства параметров:
   - Архитектура сети (количество слоёв, размеры слоёв, функции активации)
   - Параметры компиляции (оптимизаторы, функции потерь)
   - Количество эпох обучения в диапазоне от min_epoch до max_epoch с шагом epoch_step
2. Для каждой конфигурации создаётся и обучается НС.
3. Если значение метрики текущей НС лучше текущего лучшего значения, то обновляются best_metric_value, best_epoch, best_loss_func, best_opt и best_net.

Функция возвращает: best_metric_value, best_epoch, best_loss_func, best_opt, best_net